In [ ]:
from pathlib import Path
import re
import pandas as pd

CORPUS_DIR = Path(r"C:\Users\gutti\Downloads\corpus")   # output folder of beloit_ingest.py
CHUNK_WORDS = 750                                                # max words per chunk
HEADER_RULE = "-" * 60                                           # line that ends each file's header

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

In [2]:
def parse_document(path):
    """Return (metadata dict, body text) for one corpus file."""
    raw = path.read_text(encoding="utf-8").replace("\r\n", "\n")   # files may have Windows line endings
    if HEADER_RULE not in raw:
        raise ValueError("no header rule (a line of 60 dashes) found")
    head, body = raw.split(HEADER_RULE, 1)

    meta = {}
    for line in head.splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            meta[key.strip().lower()] = value.strip()

    body = body.strip()
    if not body:
        raise ValueError("body text is empty")
    return meta, body


def load_corpus(corpus_dir):
    if not corpus_dir.exists():
        raise FileNotFoundError(f"corpus folder not found: {corpus_dir}")
    docs, problems = [], []
    for path in sorted(corpus_dir.glob("*.txt")):
        try:
            meta, body = parse_document(path)
        except Exception as e:
            problems.append((path.name, str(e)))
            continue
        docs.append({
            "doc_id": meta.get("id", path.stem),
            "title": meta.get("title", ""),
            "source": meta.get("source", ""),
            "kind": meta.get("kind", ""),
            "file": path.name,
            "text": body,
            "words": len(body.split()),
        })
    return docs, problems


docs, problems = load_corpus(CORPUS_DIR)
print(f"Loaded {len(docs)} documents from {CORPUS_DIR}")
if problems:
    print(f"\nSkipped {len(problems)} file(s):")
    for name, why in problems:
        print(f"  {name}: {why}")

Loaded 142 documents from C:\Users\gutti\Downloads\corpus


In [3]:
def chunk_text(text, max_words=CHUNK_WORDS):
    """
    Split text into consecutive pieces of at most `max_words` words.
    Pieces are cut from the original string, so line breaks and paragraph breaks
    inside a chunk are kept exactly as they were in the file.
    """
    words = list(re.finditer(r"\S+", text))
    if len(words) <= max_words:
        return [text.strip()]
    pieces = []
    for start in range(0, len(words), max_words):
        group = words[start:start + max_words]
        pieces.append(text[group[0].start():group[-1].end()])
    return pieces


def chunk_documents(docs, max_words=CHUNK_WORDS):
    rows = []
    for d in docs:
        pieces = chunk_text(d["text"], max_words)
        for i, piece in enumerate(pieces, start=1):
            rows.append({
                "chunk_id": re.sub(r"\s+", "_", f"{d['doc_id']}__{i:02d}"),
                "doc_id": d["doc_id"],
                "title": d["title"],
                "source": d["source"],
                "kind": d["kind"],          # <-- added: carries the page range
                "chunk_no": i,
                "n_chunks": len(pieces),
                "words": len(piece.split()),
                "text": piece,
            })
    return pd.DataFrame(rows)


chunks = chunk_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

142 documents -> 962 chunks


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import Normalizer
import numpy as np
import joblib

INDEX_FILE = CORPUS_DIR.parent / "index" / "tfidf_index.joblib"

embedder = Pipeline([
    ("features", FeatureUnion([
        ("words", TfidfVectorizer(ngram_range=(1, 2), stop_words="english",
                                  sublinear_tf=True, strip_accents="unicode")),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                  sublinear_tf=True, min_df=2, strip_accents="unicode")),
    ])),
    ("unit_length", Normalizer()),
])

matrix = embedder.fit_transform(chunks["text"])      # one row per chunk, same order as `chunks`
print(f"embedded {matrix.shape[0]} chunks into {matrix.shape[1]:,} features "
      f"({matrix.nnz / matrix.shape[0]:,.0f} non-zero values per chunk)")

embedded 962 chunks into 116,949 features (1,898 non-zero values per chunk)


In [5]:
INDEX_FILE.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"embedder": embedder, "matrix": matrix, "chunks": chunks}, INDEX_FILE)
print(f"saved index to {INDEX_FILE}  ({INDEX_FILE.stat().st_size / 1e6:.1f} MB)")

saved index to C:\Users\gutti\Downloads\index\tfidf_index.joblib  (26.4 MB)


In [6]:
N_CHUNKS = 3               # chunks kept from the first search
N_PARAGRAPHS = 3           # paragraphs returned
MIN_PARAGRAPH_WORDS = 20   # shorter pieces are joined to the paragraph that follows


def split_paragraphs(text, min_words=MIN_PARAGRAPH_WORDS):
    parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    merged, carry = [], ""
    for p in parts:
        p = f"{carry}\n{p}" if carry else p
        if len(p.split()) < min_words:
            carry = p
        else:
            merged.append(p)
            carry = ""
    if carry:                                   # short piece at the very end
        if merged:
            merged[-1] += "\n" + carry
        else:
            merged.append(carry)
    return merged


def search(question, n_chunks=N_CHUNKS, n_paragraphs=N_PARAGRAPHS):
    q = embedder.transform([question])
    chunk_scores = (matrix @ q.T).toarray().ravel()
    best = np.argsort(chunk_scores)[::-1][:n_chunks]

    rows = []
    for i in best:
        c = chunks.iloc[i]
        for p in split_paragraphs(c["text"]):
            rows.append({"chunk_id": c["chunk_id"], "title": c["title"],
                         "kind": c.get("kind", ""),          # <-- fixed: was c["kind"]
                         "source": c["source"], "paragraph": p})
    scores = (embedder.transform([r["paragraph"] for r in rows]) @ q.T).toarray().ravel()
    out = pd.DataFrame(rows).assign(score=scores)
    return out.sort_values("score", ascending=False).head(n_paragraphs).reset_index(drop=True)


def show(question, **kw):
    print(f"QUESTION: {question}\n")
    for rank, r in search(question, **kw).iterrows():
        print(f"[{rank + 1}] score {r['score']:.3f}  |  {r['title']}  |  {r['kind']}  |  {r['chunk_id']}")
        print(r["paragraph"], "\n")

In [7]:
q1 = "What weapons are prohibited on campus?"
show(q1)

QUESTION: What weapons are prohibited on campus?

[1] score 0.303  |  Student Handbook Policies and Regulations N  |  pdf pages 113-114  |  hb_section3_n__03
could be investigated as a bias incident, threat, hate
speech, or other additional violation of campus policy.
O. Weapons
The College is private property and reserves the right to establish standards that exceed those permitted
by local, state, or federal law. Accordingly, the possession, carrying, storage, or use of weapons is strictly
prohibited on College-owned or controlled property, regardless of whether the weapon is legal or an
individual holds a permit or license. 

[2] score 0.232  |  Student Handbook Judicial Processes  |  pdf pages 114-118  |  hb_section4__01
Note regarding graffiti: “Graffiti” is defined as any unauthorized writing, drawing or marking
created on any campus structure, public or private. Regardless of whether or not graffiti is written
in permanent materials, it will be considered a form of vandalism. If

In [8]:
q2="Can I drink in any academic buildings ?"
show(q2)

QUESTION: Can I drink in any academic buildings ?

[1] score 0.173  |  Student Handbook Policies and Regulations E  |  pdf pages 55-56  |  hb_section3_e__02
endanger the safety of individuals, or destroy property. In any public demonstration, organizers or
student speakers may not speak for the College as a whole, only for themselves. Due to the
importance of academic and administrative buildings in ensuring the normal operations of the
College, demonstration and protest activity inside academic and administrative buildings is
prohibited. Demonstrations and protests, regardless of size, must maintain the ability to allow
emergency access and egress into and out of buildings, offices, and other areas at all times for
safety purposes. 

[2] score 0.166  |  Student Handbook Policies and Regulations B1  |  pdf pages 28-33  |  hb_section3_b1__01
and trust in students to make decisions about drinking alcohol that are in their best interest of
safety, health, social interactions and meeting a

In [9]:
q3="Can I use spraychalk on beloit campus/sidewalks?"
show(q3)

QUESTION: Can I use spraychalk on beloit campus/sidewalks?

[1] score 0.204  |  Student Handbook Policies and Regulations D2  |  pdf pages 50-53  |  hb_section3_d2__06
3. Publicity, Solicitation, Signs
Chalking Policy
The use of water-soluble chalk is permitted on horizontal surfaces exposed to weather. Areas
where chalking is not permitted include buildings, vertical surfaces (walls), painted surfaces,
porches, wooden surfaces, in the MI tunnels, under the walkways of the ’64 Halls, and other
surfaces not exposed to weather. Chalking is not permitted on the steps, ramps or red brick area
of Middle College. The use of spray-on chalk or any other formulation (including the mixing of
chalk and water as a paint) is prohibited on the Beloit College campus and sidewalks. 

[2] score 0.204  |  Student Handbook Policies and Regulations D3  |  pdf pages 53-55  |  hb_section3_d3__01
3. Publicity, Solicitation, Signs
Chalking Policy
The use of water-soluble chalk is permitted on horizontal surfa